# Brevitas code
## Define quantized model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import brevitas.nn as qnn

class CNetPlusScalar(nn.Module):
    """
    CNetPlusScalar inherits from CNet but rewrites the forward method by introducing a scalar value to the output.
    The difference is that x is now a dictionary with the key 'image' and 'scalar'.
    old CNN applicable to 512x512 images

    Compared to the original CNet, this model assumes only one extra feature (scalar) to be concatenated to the output of the last convolutional layer and one input channel. This extra feature is the background value. Furthermore, the original leaky ReLU activation function is replaced by the ReLU activation function.

    In this quantized version, the convolutional layers are replaced by quantized convolutional layers and the linear layers are replaced by quantized linear layers. Furthermore, a quantized identity layer is added at the beginning of the network to quantize the input image.
    """
    def __init__(self):
        super().__init__()
        self.quant_inp_img = qnn.QuantIdentity(return_quant_tensor=True)
        self.quant_inp_scalar = qnn.QuantIdentity(return_quant_tensor=True)
        self.pool = nn.MaxPool2d(2, 2) 
        self.conv1 = qnn.QuantConv2d(1, 3, 5, return_quant_tensor=True) 
        self.conv2 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True) 
        self.conv3 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True) 
        self.conv4 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True)
        self.conv5 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True)
        self.conv6 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True)
        self.conv7 = qnn.QuantConv2d(3, 3, 3, return_quant_tensor=True)
        self.fc1 = qnn.QuantLinear(3 * 2 * 2, 30, return_quant_tensor=True)
        self.fc2 = qnn.QuantLinear(30, 30, return_quant_tensor=True)
        self.fc3 = qnn.QuantLinear(30+1, 1, return_quant_tensor=True)

    def forward(self, image, background_scalar):
        image = self.quant_inp_img(image)
        x = self.pool(F.relu(self.conv1(image))) 
        x = self.pool(F.relu(self.conv2(x))) 
        x = self.pool(F.relu(self.conv3(x))) 
        x = self.pool(F.relu(self.conv4(x))) 
        x = self.pool(F.relu(self.conv5(x)))
        x = self.pool(F.relu(self.conv6(x)))
        x = self.pool(F.relu(self.conv7(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.quant_inp_scalar(x)
        background_scalar = self.quant_inp_scalar(background_scalar)
        x = torch.cat((x, background_scalar), dim=1) # adding the scalar value
        x = self.fc3(x)
        return x

model = CNetPlusScalar()

## Load pre-trained weights

In [ ]:
import brevitas.config as config
config.IGNORE_MISSING_KEYS = True
model.load_state_dict(torch.load("pre_trained_w.pt", weights_only=True))

## Finetune pre-trained model

In [ ]:
# Generate random test inputs
num_samples = 100
dataset = torch.rand((1, 1, 512, 512), dtype=torch.float32)

## Test model

In [ ]:
# WIP
model.eval() 

with torch.no_grad():
    scalar = 0
    for image in dataset:
        scalar = model(image, scalar)

## Export to ONNX

In [ ]:
import onnx
from finn.util.visualization import showSrc, showInNetron
from finn.util.basic import make_build_dir
import os
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

build_dir = os.environ["FINN_BUILD_DIR"]
onnx_path = build_dir+"/Cnet.onnx"

# Export to ONNX
export_qonnx(
    model, export_path=onnx_path, input_t=torch.randn((1, 3, 128, 256), dtype=torch.float32)
)

# clean-up
qonnx_cleanup(onnx_path, out_file=onnx_path)

# ModelWrapper
model = ModelWrapper(onnx_path)
# Setting the input datatype explicitly because it doesn't get derived from the export function
model.set_tensor_datatype(model.graph.input[0].name, DataType["FLOAT32"])
model = model.transform(ConvertQONNXtoFINN())
model.save(onnx_path)

print("Model saved to %s" % onnx_path)

# FINN

In [ ]:
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

onnx_path = build_dir+"/Cnet_tidy.onnx"
model.save(onnx_path)
showInNetron(onnx_path)

## Conversion to HW

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

estimates_output_dir = "output_estimates_only"

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,
    fpga_part           = "xczu7ev-ffvc1156-2-e",
    steps               = build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [ ]:
%%time
build.build_dataflow_cfg(onnx_path, cfg_estimates)

In [ ]:
showInNetron("output_estimates_only/intermediate_models/step_create_dataflow_partition.onnx")

In [ ]:
final_output_dir = "output_final"

#Delete previous run results if exist
if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted!")

cfg = build.DataflowBuildConfig(
    output_dir          = final_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 5.0,
    board               = "ZCU104",
    shell_flow_type     = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.BITFILE,
        build_cfg.DataflowOutputType.PYNQ_DRIVER,
        build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ]
)

In [ ]:
#build.build_dataflow_cfg(onnx_path, cfg)